# Equity options (Black–76 + Monte Carlo benchmark)

**Prerequisites:** **06**. **`equity_option`** with **`black76`**; compare to **`finstack.monte_carlo`** European pricer as an independent simulation check.


## Concept

- **Black–76** on forwards: discounting, dividend yield, vol surface.
- **Delta** (and other Greeks) come from the analytical pricer when registered.
- **Monte Carlo** reproduces the same risk-neutral expectation subject to simulation noise.


In [ ]:
import json
from datetime import date

from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json
from finstack.core.market_data import DiscountCurve, MarketContext
from finstack.monte_carlo import EuropeanPricer

print("Imports OK (equity options).")


## Instrument JSON


In [ ]:
AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

equity_option = json.dumps(
    {
        "type": "equity_option",
        "spec": {
            "id": "SPX-CALL-4500",
            "underlying_ticker": "SPX",
            "strike": 4500.0,
            "option_type": "call",
            "exercise_style": "european",
            "expiry": "2026-06-15",
            "notional": {"amount": 100.0, "currency": "USD"},
            "day_count": "Act365F",
            "settlement": "cash",
            "discount_curve_id": "USD-OIS",
            "spot_id": "EQUITY-SPOT",
            "vol_surface_id": "EQUITY-VOL",
            "div_yield_id": "EQUITY-DIVYIELD",
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

try:
    validate_instrument_json(equity_option)
    print("equity_option JSON OK")
except Exception as exc:
    print("validate:", type(exc).__name__, exc)


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
mc = MarketContext().insert(ois)
state = json.loads(mc.to_json())
exp_eq = [0.25, 0.5, 1.0, 2.0]
strikes_eq = [4000.0, 4300.0, 4500.0, 4700.0, 5000.0]
state["surfaces"] = [
    {
        "id": "EQUITY-VOL",
        "expiries": exp_eq,
        "strikes": strikes_eq,
        "secondary_axis": "strike",
        "interpolation_mode": "vol",
        "vols_row_major": [0.22] * (len(exp_eq) * len(strikes_eq)),
    }
]
state.setdefault("prices", {})
state["prices"]["EQUITY-SPOT"] = {"price": {"amount": 4500.0, "currency": "USD"}}
state["prices"]["EQUITY-DIVYIELD"] = {"unitless": 0.015}
market_json = json.dumps(state)
print("surfaces:", len(json.loads(market_json)["surfaces"]))


## Pricing


In [ ]:
try:
    out = price_instrument_with_metrics(
        equity_option, market_json, AS_OF_STR, model="black76", metrics=["delta"]
    )
    vr = ValuationResult.from_json(out)
    print("JSON black76:", vr)
    print("delta:", vr.get_metric("delta"))
except Exception as exc:
    print("black76:", type(exc).__name__, exc)


## Metrics & MC cross-check


In [ ]:
# Independent GBM Monte Carlo on a stylized underlying (not the JSON pricer MC path).
spot = 4500.0
strike = 4500.0
r = 0.045
q = 0.015
vol = 0.22
T = 500.0 / 365.0
pricer = EuropeanPricer(num_paths=40_000, seed=42)
mc_res = pricer.price_call(spot=spot, strike=strike, rate=r, div_yield=q, vol=vol, expiry=T)
print("MC call (per unit notional scale): mean=", mc_res.mean.amount, "stderr=", mc_res.stderr)
print("Analytical black76 price from JSON pipeline printed above; MC uses flat vol", vol, "T=", round(T, 4))


## Takeaways

- **`black76`** is the JSON model key for vanilla equity options here.
- **`finstack.monte_carlo.EuropeanPricer`** gives a simulation benchmark; align vol, div yield, rate, and horizon when comparing.
- JSON **`price_instrument_with_metrics`** does not currently register `monte_carlo_gbm` for `equity_option` in this build.
